# Custom word embeddings on SNLI (V=1000)

1. BPE tokenizer with 1000 tokens, trained on SNLI's training sentences
2. Embedding model: each token predicts the token before it and the token after it
3. Accuracy on validation sentences + nearest neighbours

Model code: `src/embedding_models.py`. Tokenizer: `src/tokenizer.py`.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

from src.dataset import convert_to_dataframe, load_snli_dataset
from src.embedding_models import nearest, train_embeddings
from src.tokenizer import UNK_TOKEN, train_tokenizer

V = 1000          # BPE vocab size
D = 100           # embedding size
EPOCHS = 3
MIN_FREQUENCY = 2

## Data
Unique sentences only: each premise appears ~3 times in SNLI (once per label), so without deduplication premises would be over-weighted.
Everything is trained on `train_sents`; `val_sents` is only used to measure.

In [2]:
train_df, val_df, test_df = convert_to_dataframe(load_snli_dataset())


def unique_sentences(df):
    sents = pd.concat([df["premise"], df["hypothesis"]]).dropna().astype(str)
    return list(dict.fromkeys(sents))  # dedupe, keep order


train_sents = unique_sentences(train_df)
val_sents = unique_sentences(val_df)
print(f"unique sentences  train {len(train_sents):,}  val {len(val_sents):,}")

unique sentences  train 628,646  val 12,982


## Tokenizer (BPE, V=1000, lowercased)
`▁` marks the start of a word: `▁dog` is a whole word, `ing` (no `▁`) is a piece glued to the previous one.

In [3]:
tok = train_tokenizer(train_sents, V, min_frequency=MIN_FREQUENCY, lowercase=True)

for s in ["A man is skateboarding down a railing.", "Two dogs are playing in the snow."]:
    print(s, "->", tok.encode(s).tokens)




A man is skateboarding down a railing. -> ['▁a', '▁man', '▁is', '▁skateboard', 'ing', '▁down', '▁a', '▁ra', 'il', 'ing.']
Two dogs are playing in the snow. -> ['▁two', '▁dogs', '▁are', '▁playing', '▁in', '▁the', '▁snow.']


### Coverage on validation sentences
- **tokens/sentence**: sequence length the encoders will see
- **tokens/word**: pieces per word (1.0 = every word whole)
- **% words whole**: running words (tokens) / distinct words (types) that are a single vocab entry
- **% UNK**: tokens that could not be represented

Last run (V=1000): 1.49 pieces/word, 74% of running words whole, 13.7 tokens/sentence, 0% UNK.
Punctuation sticks to the last word (`▁snow.`), since Metaspace only splits on spaces.

In [4]:
def tokenizer_stats(tok, sentences):
    enc = tok.encode_batch(sentences)
    ids = np.concatenate([e.ids for e in enc])
    words = Counter(w for s in sentences for w in s.lower().split())
    types = list(words)
    pieces = np.array([len(e.ids) for e in tok.encode_batch(types)])
    freq = np.array([words[w] for w in types])
    return {
        "tokens/sentence": np.mean([len(e.ids) for e in enc]),
        "tokens/word": (pieces * freq).sum() / freq.sum(),
        "% words whole (tokens)": 100 * freq[pieces == 1].sum() / freq.sum(),
        "% words whole (types)": 100 * (pieces == 1).mean(),
        "% UNK": 100 * (ids == tok.token_to_id(UNK_TOKEN)).mean(),
    }


pd.Series(tokenizer_stats(tok, val_sents)).round(2)

tokens/sentence           13.71
tokens/word                1.49
% words whole (tokens)    74.17
% words whole (types)      4.52
% UNK                      0.00
dtype: float64

## Embedding model: one token back, one token ahead
`nn.Embedding(V, D)` → two `nn.Linear(D, V)` heads (previous token, next token). Loss = CE(prev) + CE(next); sentence edges get target `-100`, which the loss ignores.
The first loss should be about 2·ln(1000) ≈ 13.8 (random guessing). After each epoch it reports loss and top-1 / top-5 accuracy on the validation sentences.

In [5]:
emb_model, emb_tok, history = train_embeddings(
    train_sents, d=D, epochs=EPOCHS, val_sentences=val_sents, tokenizer=tok
)

8,313,306 training tokens | V=1000 d=100 | 302,000 params | device=mps
loss at random init should be about 2*ln(V) = 13.82
  epoch 1 step    200  loss 11.694
  epoch 1 step    400  loss 10.459
  epoch 1 step    600  loss 9.777
  epoch 1 step    800  loss 9.358
  epoch 1 step   1000  loss 9.072
  epoch 1 step   1200  loss 8.867
  epoch 1 step   1400  loss 8.710
  epoch 1 step   1600  loss 8.585
  epoch 1 step   1800  loss 8.484
  epoch 1 step   2000  loss 8.400
epoch 1: train loss 8.389 | val loss 7.592 | prev top1 22.8% top5 48.7% | next top1 20.9% top5 45.4%  (13s)
  epoch 2 step    200  loss 7.610
  epoch 2 step    400  loss 7.601
  epoch 2 step    600  loss 7.592
  epoch 2 step    800  loss 7.585
  epoch 2 step   1000  loss 7.579
  epoch 2 step   1200  loss 7.573
  epoch 2 step   1400  loss 7.567
  epoch 2 step   1600  loss 7.561
  epoch 2 step   1800  loss 7.556
  epoch 2 step   2000  loss 7.551
epoch 2: train loss 7.550 | val loss 7.474 | prev top1 22.8% top5 49.1% | next top1 20.

### Training curve and accuracy
`top1` = the head's best guess is the real neighbour; `top5` = it is among the 5 best guesses.
Top-1 stays well below 100% even for a good model: after `▁a` many words are valid.
Val loss can be lower than train loss: train loss is averaged over the epoch while the model was still improving, val loss is measured at the end.

In [6]:
pd.DataFrame(history).set_index("epoch").round(3)

,train_loss,val_loss,val_prev_top1,val_prev_top5,val_next_top1,val_next_top5
epoch,,,,,,
1,8.389,7.592,22.760,48.710,20.901,45.401
2,7.550,7.474,22.796,49.064,20.773,45.765
3,7.475,7.428,22.816,49.222,20.774,45.943


## What did it learn? Nearest neighbours (cosine similarity)
Only words that are a single token at V=1000 can be looked up; others print how they get split.

Last run: colours, numbers, people and verb forms cluster well, but opposites end up as nearest neighbours
(`sitting` → `standing`, `outside` → `inside`) because they fit the same slot. That is exactly what decides
contradiction in NLI. `not` learned nothing useful.

In [7]:
for w in ["man", "woman", "dog", "red", "sitting", "standing", "running", "playing", "two", "outside", "not"]:
    try:
        print(f"{w:>9}: {nearest(w, emb_model, emb_tok, k=8)}")
    except KeyError as e:
        print(f"{w:>9}: {e}")

      man: [('▁guy', 0.491), ('▁male', 0.478), ('▁lady', 0.412), ('▁worker', 0.405), ('▁person', 0.377), ('▁mother', 0.37), ('▁grass.', 0.348), ('▁woman', 0.329)]
    woman: [('▁couple', 0.453), ('▁person', 0.449), ('▁women', 0.438), ('▁lady', 0.421), ('▁girl', 0.402), ('▁guy', 0.376), ('▁boy', 0.365), ('▁suit', 0.36)]
      dog: [('▁horse', 0.405), ('▁family', 0.377), ('▁bird', 0.356), ('▁camera.', 0.351), ('▁jump', 0.308), ('▁boys', 0.296), ('▁truck', 0.295), ('▁dogs', 0.292)]
      red: [('▁blue', 0.51), ('▁pink', 0.498), ('▁white', 0.476), ('▁yellow', 0.456), ('▁green', 0.422), ('▁orange', 0.385), ('▁brown', 0.334), ('▁long', 0.326)]
  sitting: [('▁standing', 0.52), ('▁working', 0.406), ('▁laying', 0.39), ('▁sits', 0.384), ('▁sit', 0.383), ('▁driving', 0.382), ('▁lay', 0.351), ('▁rides', 0.34)]
 standing: [('▁sitting', 0.52), ('▁sleeping', 0.435), ('▁stands', 0.415), ('▁laying', 0.401), ('▁reading', 0.393), ('▁sits', 0.374), ('▁sit', 0.359), ('▁stand', 0.341)]
  running: [('▁eating

## The embedding matrix
Row i = vector of token id i in `tok`; row 0 = `[PAD]` = zeros. This is what the pooling / GRU / LSTM models take as input.

In [8]:
W = emb_model.embedding_weights()
print(W.shape, W[0].abs().sum().item())

torch.Size([1000, 100]) 0.0


## NLI with the custom embeddings: mean pool + Part 1 MLPs
Code: `src/part2_models.py`.
- Each sentence → BPE tokens → custom embedding of each token → **average** → one 100-dim vector
- Pair → `[premise ; hypothesis]` (200 numbers) → one of last week's MLPs → entailment / neutral / contradiction
  - **MLPSimple**: 1 hidden layer (128) on `[u ; v]`; dropout 0.1, AdamW lr 1e-4, weight decay 1e-4, batch 64
  - **MLPNLI**: 2 hidden layers (128, 64) on `[u, v, |u-v|, u·v]`; dropout 0.3, AdamW lr 3e-4, weight decay 1e-6, batch 128, grad clipping 1.0
- Embeddings are frozen; the epoch with the best val accuracy is kept.

Last week for comparison (50k subsample, 5-fold CV): bag-of-words + MLPSimple 66.4%, + MLPNLI 66.7%; MPNet sentence embeddings + MLPSimple 72.3%, + MLPNLI 75.2%.

In [9]:
from src.part2_models import accuracy, pair_features, train_mlp

W_custom = emb_model.embedding_weights()  # (1000, 100)

X_train, y_train = pair_features(train_df, tok, W_custom)
X_val, y_val = pair_features(val_df, tok, W_custom)
X_test, y_test = pair_features(test_df, tok, W_custom)
print(X_train.shape, X_val.shape, X_test.shape)

torch.Size([549367, 200]) torch.Size([9842, 200]) torch.Size([9824, 200])


In [10]:
mlp_simple, hist_simple = train_mlp(X_train, y_train, X_val, y_val, model="simple", epochs=20)

epoch  1: val acc 54.30%  (2s)
epoch  2: val acc 56.08%  (2s)
epoch  3: val acc 58.49%  (2s)
epoch  4: val acc 59.27%  (2s)
epoch  5: val acc 60.91%  (2s)
epoch  6: val acc 61.69%  (2s)
epoch  7: val acc 62.56%  (2s)
epoch  8: val acc 62.69%  (2s)
epoch  9: val acc 62.44%  (2s)
epoch 10: val acc 63.70%  (2s)
epoch 11: val acc 63.71%  (2s)
epoch 12: val acc 63.09%  (2s)
epoch 13: val acc 63.79%  (2s)
epoch 14: val acc 63.94%  (2s)
epoch 15: val acc 64.09%  (2s)
epoch 16: val acc 64.23%  (2s)
epoch 17: val acc 63.84%  (2s)
epoch 18: val acc 64.65%  (2s)
epoch 19: val acc 64.65%  (2s)
epoch 20: val acc 63.94%  (2s)


In [11]:
mlp_nli, hist_nli = train_mlp(X_train, y_train, X_val, y_val, model="nli", epochs=20)

epoch  1: val acc 59.41%  (3s)
epoch  2: val acc 61.75%  (3s)
epoch  3: val acc 62.48%  (3s)
epoch  4: val acc 62.95%  (3s)
epoch  5: val acc 63.11%  (3s)
epoch  6: val acc 63.60%  (3s)
epoch  7: val acc 64.13%  (3s)
epoch  8: val acc 64.33%  (3s)
epoch  9: val acc 63.51%  (3s)
epoch 10: val acc 64.35%  (3s)
epoch 11: val acc 64.73%  (3s)
epoch 12: val acc 64.54%  (3s)
epoch 13: val acc 64.46%  (3s)
epoch 14: val acc 64.85%  (3s)
epoch 15: val acc 64.66%  (3s)
epoch 16: val acc 64.71%  (3s)
epoch 17: val acc 64.83%  (3s)
epoch 18: val acc 65.52%  (3s)
epoch 19: val acc 65.20%  (3s)
epoch 20: val acc 65.40%  (3s)


### Accuracy (%) on train / val / test

In [12]:
scores = pd.DataFrame({
    name: {
        "train": accuracy(m, X_train, y_train),
        "val": accuracy(m, X_val, y_val),
        "test": accuracy(m, X_test, y_test),
    }
    for name, m in [("custom emb + MLPSimple", mlp_simple), ("custom emb + MLPNLI", mlp_nli)]
}).T.round(2)
scores

,train,val,test
custom emb + MLPSimple,65.16,64.65,64.61
custom emb + MLPNLI,66.86,65.52,64.26


## LSTM encoder (basic version)
Code: `src/part2_models.py` (`LSTMPairClassifier`, `train_lstm`).
- Each sentence → token ids → custom embeddings (**frozen**) → single-layer LSTM (hidden 128) → final hidden state = sentence vector
- Same LSTM for premise and hypothesis → `[u ; v]` → Part 1 MLPNLI → label
- Adam lr 1e-3, batch 128, gradient clipping 1.0, 5 epochs, best-val epoch kept

Unlike mean pooling, the LSTM reads words in order, so it can tell *where* a word appears.
Design choices (hidden size, layers, bidirectional, trainable embeddings, data size) come later.

In [13]:
from src.part2_models import lstm_accuracy, token_tensors, train_lstm

PAD_ID = tok.token_to_id("[PAD]")
train_tok = token_tensors(train_df, tok, pad_id=PAD_ID)
val_tok = token_tensors(val_df, tok, pad_id=PAD_ID)
test_tok = token_tensors(test_df, tok, pad_id=PAD_ID)
print(train_tok["p_ids"].shape, val_tok["p_ids"].shape, test_tok["p_ids"].shape)

torch.Size([549367, 64]) torch.Size([9842, 64]) torch.Size([9824, 64])


In [14]:
lstm, hist_lstm = train_lstm(train_tok, val_tok, W_custom, pad_id=PAD_ID, epochs=5)

549,367 training pairs | 191,875 trainable params | device=mps
epoch 1: train loss 0.773 | val acc 73.33%  (368s)
epoch 2: train loss 0.638 | val acc 75.66%  (356s)
epoch 3: train loss 0.591 | val acc 76.74%  (358s)
epoch 4: train loss 0.562 | val acc 77.46%  (320s)
epoch 5: train loss 0.540 | val acc 77.43%  (315s)


In [15]:
scores.loc["custom emb + LSTM + MLPNLI"] = [
    lstm_accuracy(lstm, train_tok), lstm_accuracy(lstm, val_tok), lstm_accuracy(lstm, test_tok)
]
scores.round(2)

,train,val,test
custom emb + MLPSimple,65.16,64.65,64.61
custom emb + MLPNLI,66.86,65.52,64.26
custom emb + LSTM + MLPNLI,79.44,77.46,77.05
